# Xây dựng mô hình Naïve ngây thơ trên tập dữ liệu hành vi của khách hàng lấy tại https://www.kaggle.com/code/arezalo/customer-behaviour-prediction-naive-bayes



## 1. Mục tiêu

- Làm quen với thuật toán Naïve Bayes (GaussianNB) cho bài toán phân loại nhị phân.
- Áp dụng mô hình để dự đoán khả năng mua hàng của khách hàng dựa trên các thuộc tính: giới tính, tuổi, mức lương ước tính.
- Đánh giá hiệu quả mô hình bằng các chỉ số: Accuracy, Confusion Matrix, Precision, Recall, F1-score.

## 2. Import thư viện và nạp dữ liệu

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Đọc dữ liệu
data = pd.read_csv('Customer_Behaviour.csv')

print("5 dòng đầu tiên của dữ liệu:")
print(data.head())

print("Thông tin dữ liệu:")
print(data.info())

5 dòng đầu tiên của dữ liệu:
    User ID  Gender  Age  EstimatedSalary  Purchased
0  15624510    Male   19            19000          0
1  15810944    Male   35            20000          0
2  15668575  Female   26            43000          0
3  15603246  Female   27            57000          0
4  15804002    Male   19            76000          0
Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   User ID          400 non-null    int64 
 1   Gender           400 non-null    object
 2   Age              400 non-null    int64 
 3   EstimatedSalary  400 non-null    int64 
 4   Purchased        400 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 15.8+ KB
None


**Nhận xét:**
- Dữ liệu gồm 400 dòng và 5 cột: User ID, Gender, Age, EstimatedSalary, Purchased.
- Không có giá trị bị thiếu (non-null = 400 cho tất cả các cột).
- 'User ID' là mã định danh, không tham gia vào quá trình dự đoán.
- 'Gender' là biến phân loại dạng chuỗi (Male/Female) cần mã hoá.
- 'Purchased' là biến mục tiêu: 0 - Không mua, 1 - Có mua.

## 3. Tiền xử lý dữ liệu

In [2]:
# (a) Bỏ cột không cần thiết (ID chỉ là định danh)
data = data.drop(columns=['User ID'])

# (b) Mã hoá cột "Gender" sang số
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender']) # Ví dụ: Female -> 0, Male -> 1 (tuỳ bộ mã hoá)

# (c) Tách đặc trưng (x) và nhãn (y)
x = data.drop('Purchased', axis=1)
y = data['Purchased']

In [3]:
print("\nDữ liệu sau tiền xử lý (5 dòng đầu):")
print(data.head())


Dữ liệu sau tiền xử lý (5 dòng đầu):
   Gender  Age  EstimatedSalary  Purchased
0       1   19            19000          0
1       1   35            20000          0
2       0   26            43000          0
3       0   27            57000          0
4       1   19            76000          0


**Nhận xét:**
- Đã loại bỏ cột 'User ID' vì không mang ý nghĩa trong dự đoán.
- Cột 'Gender' đã được mã hoá thành dạng số (0/1), phù hợp với mô hình học máy.
- Tập đặc trưng X gồm 3 cột: Gender, Age, EstimatedSalary.
- Biến mục tiêu y là cột 'Purchased'.

In [4]:
# (d) Chia dữ liệu train/test (80% train – 20% test)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# (e) Chuẩn hoá dữ liệu (nên làm với các mô hình dùng phân phối chuẩn như GaussianNB)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

print("Đã chia dữ liệu thành tập huấn luyện và kiểm tra (80/20) và chuẩn hoá các thuộc tính số")

Đã chia dữ liệu thành tập huấn luyện và kiểm tra (80/20) và chuẩn hoá các thuộc tính số


**Nhận xét:**
- Việc chuẩn hoá (StandardScaler) giúp các thuộc tính Age và EstimatedSalary
  có cùng thang đo (mean ~ 0, std ~ 1), giúp GaussianNB hoạt động ổn định hơn.

## 4. Xây dựng mô hình Naïve Bayes

In [5]:
# Khởi tạo và huấn luyện mô hình Gaussian Naïve Bayes
classifier = GaussianNB()
classifier.fit(x_train_scaled, y_train)

,priors,None
,var_smoothing,1e-09


- GaussianNB giả định mỗi thuộc tính (Age, EstimatedSalary, Gender) tuân theo phân phối chuẩn trong từng lớp (Purchased = 0 hoặc 1).
- Mô hình Naïve Bayes đơn giản, tốc độ huấn luyện nhanh, phù hợp cho bài toán phân loại cơ bản.

## 5. Đánh giá mô hình

In [6]:
# Dự đoán trên tập test
y_pred = classifier.predict(x_test_scaled)

# Tính các chỉ số đánh giá
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classif_report = classification_report(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classif_report)


Accuracy: 0.9375

Confusion Matrix:
[[50  2]
 [ 3 25]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95        52
           1       0.93      0.89      0.91        28

    accuracy                           0.94        80
   macro avg       0.93      0.93      0.93        80
weighted avg       0.94      0.94      0.94        80



**Nhận xét chi tiết về kết quả:**

**1. Độ chính xác (Accuracy)**
- Mô hình đạt độ chính xác khoảng 94% trên tập kiểm tra. Điều này có nghĩa là trong 100 khách hàng, mô hình dự đoán đúng khoảng 94 trường hợp. Với một mô hình đơn giản như Naïve Bayes, đây là kết quả rất khả quan, cho thấy mô hình phù hợp với bài toán phân loại nhị phân trên dữ liệu hành vi khách hàng.

**2. Phân tích ma trận nhầm lẫn (Confusion Matrix)**

Ma trận nhầm lẫn cho biết mức độ mô hình dự đoán đúng hoặc sai từng lớp:
- True Negative (TN): số khách hàng không mua và mô hình dự đoán đúng là không mua.
- False Positive (FP): số khách hàng không mua nhưng mô hình dự đoán nhầm là mua.
- False Negative (FN): số khách hàng có mua nhưng mô hình dự đoán nhầm là không mua.
- True Positive (TP): số khách hàng có mua và mô hình dự đoán đúng là có mua.

→ Kết quả cho thấy số lượng nhầm lẫn FP và FN đều thấp, chứng tỏ mô hình phân biệt tốt giữa 2 lớp “mua” và “không mua”.

**3. Nhận xét từ Classification Report**

- Precision của cả hai lớp đều nằm trong khoảng 0.93 – 0.94. Điều này cho thấy khi mô hình đưa ra dự đoán, khả năng dự đoán đúng là rất cao.
- Recall cũng đạt mức cao (0.89 – 0.96), cho thấy mô hình đã nhận diện đúng phần lớn các trường hợp thực tế của mỗi lớp.
- F1-score đều trên 0.90, thể hiện sự cân bằng tốt giữa precision và recall.

Nhìn chung, mô hình hoạt động ổn định và hiệu quả, không bị thiên lệch về một lớp nào.

**4. Kết luận chung**

- Mô hình Gaussian Naïve Bayes đạt hiệu suất tốt trên tập dữ liệu hành vi khách hàng.
- Độ chính xác cao, các chỉ số precision, recall và F1-score đều cho thấy mô hình đáng tin cậy.
- Đây là mô hình phù hợp làm baseline cho bài toán dự đoán khách hàng có mua hay không.
- Hiệu quả có thể được cải thiện hơn nếu bổ sung thêm các mô hình khác như Logistic Regression, SVM hoặc Random Forest, hoặc thêm nhiều thuộc tính đầu vào để tăng tính mô tả của dữ liệu.


# Kết thúc